In [1]:
import pandas as pd
import numpy as np
# Load data
df = pd.read_csv("../data/processed/nav_data.csv")

# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

# Fill missing values using forward fill
df = df.ffill()

# Verify cleaning
print("\nAfter Cleaning:")
print(df.isnull().sum())

# Display first 5 rows
print("\nFirst 5 Rows:")
print(df.head())

# Save cleaned data
df.to_csv("../data/processed/nav_data_cleaned.csv", index=False)

print("\nData cleaned successfully!")

Missing Values:
date    0
nav     0
dtype: int64

Duplicate Rows:
0

After Cleaning:
date    0
nav     0
dtype: int64

First 5 Rows:
         date       nav
0  2013-01-02  103.0059
1  2013-01-03  103.0306
2  2013-01-04  103.0619
3  2013-01-07  103.1408
4  2013-01-08  103.1663

Data cleaned successfully!


In [2]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")

print(nav.head())
print(nav.dtypes)
print(nav.shape)

   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
amfi_code      int64
date             str
nav          float64
dtype: object
(46000, 3)


In [12]:
nav['date'] = pd.to_datetime(nav['date'])

In [4]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

print(nav.head())

   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [5]:
nav['date'] = pd.to_datetime(nav['date'])

In [6]:
print(nav.dtypes)

amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object


In [7]:
nav = nav.sort_values(by=['amfi_code', 'date'])

In [8]:
nav = nav.drop_duplicates()

In [9]:
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

In [10]:
nav = nav[nav['nav'] > 0]

In [11]:
nav.to_csv("../data/processed/nav_history_cleaned.csv", index=False)

In [14]:
txn = pd.read_csv("../data/raw/08_investor_transactions.csv")

print(txn.head())
print(txn.dtypes)
print(txn.shape)

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

In [15]:
txn['transaction_type'] = txn['transaction_type'].str.upper()

txn['transaction_type'] = txn['transaction_type'].replace({
    'SIP ': 'SIP',
    'LUMP SUM': 'LUMPSUM'
})

In [16]:
txn = txn[txn['amount'] > 0]

KeyError: 'amount'

In [17]:
print(txn.columns)

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='str')


In [18]:
txn = txn[txn['amount_inr'] > 0]

In [19]:
txn['transaction_type'] = txn['transaction_type'].str.upper()

txn['transaction_type'] = txn['transaction_type'].replace({
    'SIP ': 'SIP',
    'LUMP SUM': 'LUMPSUM'
})

In [20]:
txn = txn[txn['amount_inr'] > 0]

In [21]:
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])

In [22]:
print(txn['kyc_status'].unique())

<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str


In [23]:
txn.to_csv("../data/processed/investor_transactions_cleaned.csv", index=False)

In [25]:
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [28]:
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct'
]

for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

In [27]:
print(perf.columns)

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')


In [30]:
anomalies = perf[
    (perf['expense_ratio_pct'] < 0.1) |
    (perf['expense_ratio_pct'] > 2.5)
]

print(anomalies)

Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []


In [31]:
perf = perf[
    (perf['expense_ratio_pct'] >= 0.1) &
    (perf['expense_ratio_pct'] <= 2.5)
]

In [32]:
perf.to_csv("../data/processed/scheme_performance_cleaned.csv", index=False)

In [ ]:
print(txn.head())
print(txn['transaction_type'].unique())